<a href="https://colab.research.google.com/github/mohamadfaisalbashir/Practical-Linear-Algebra/blob/main/09_Orthogonal_Matrices_and_QR_Decomposition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Orthogonal Matrices and QR Decomposition**

Orthogonal Matrices and QR Decomposition:
1. Orthogonal Matrices
2. Gram-Schmidt
3. QR Decomposition
4. Summary

## **1. Orthogonal Matrices**

An orthogonal matrix is a special type of matrix with remarkable properties. A square matrix Q is orthogonal if its columns are pairwise orthogonal (perpendicular to each other) and each column has a unit norm (length of 1). This means that when you compute the dot product of any two different columns, you get zero, and the dot product of any column with itself gives one. These properties lead to one of the most elegant relationships in linear algebra: the product of an orthogonal matrix with its transpose equals the identity matrix. In mathematical notation, this is expressed as Q^T Q = I, where Q^T is the transpose of Q and I is the identity matrix. Furthermore, this relationship implies that the transpose of an orthogonal matrix is also its inverse, giving us Q^(-1) = Q^T. This property is extraordinarily useful because computing the transpose is far simpler than computing the matrix inverse through traditional methods. Orthogonal matrices appear throughout linear algebra and its applications, including rotations in computer graphics, pure rotation transformations, and many matrix decompositions such as eigendecomposition and singular value decomposition. Their numerical stability and computational efficiency make them invaluable tools in scientific computing and data analysis.

### **1.1 Understanding QR Decomposition**

QR decomposition is one of the most important matrix factorizations in applied linear algebra. The fundamental idea is to decompose any matrix A into the product of two matrices: an orthogonal matrix Q and an upper-triangular matrix R. Mathematically, we write this as A = QR. The orthogonal matrix Q has columns that form an orthonormal basis for the column space of A, while the upper-triangular matrix R contains the coefficients that describe how to combine these orthonormal basis vectors to reconstruct the original matrix A. The beauty of this decomposition lies in the fact that it can be computed from any matrix, not just orthogonal matrices, and the process leverages the Gram-Schmidt orthogonalization procedure. Although Gram-Schmidt has theoretical importance for understanding how QR decomposition works, the actual implementation in numerical computing libraries uses more stable algorithms such as Householder reflections or Givens rotations to avoid numerical errors that can accumulate in the direct Gram-Schmidt procedure. The QR decomposition has numerous applications including solving least squares problems, computing matrix inverses, and finding eigenvalues. Its superior numerical stability compared to other methods makes it the preferred choice in many practical applications.

## **2. Gram-Schmidt**

The Gram-Schmidt orthogonalization procedure is a fundamental algorithm that transforms a set of linearly independent vectors (represented as columns of a matrix) into an orthonormal set of vectors. While Gram-Schmidt is not the most numerically stable method in practice (which is why modern implementations use Householder reflections instead), understanding Gram-Schmidt is crucial for conceptualizing how QR decomposition works and why it produces the specific structure of the R matrix. The algorithm works iteratively from left to right through the columns of the input matrix V. For each column vector v_k, the procedure first removes all components that are parallel to the previously orthogonalized columns (this is done through orthogonal vector decomposition), resulting in a vector v_k^* that is orthogonal to all previous columns. Then, this orthogonalized vector is normalized to unit length by dividing by its norm. The resulting unit vector becomes the k-th column of the orthogonal matrix Q. The first column receives special treatment since there are no previous columns to orthogonalize against; it is simply normalized to unit length. This process guarantees that each column of Q is orthogonal to all other columns and has unit norm, satisfying the definition of an orthogonal matrix. The key insight is that this column-by-column orthogonalization process naturally produces an upper-triangular structure in the R matrix, since the dot products between earlier columns in Q and later columns in A are not forced to be zero during the procedure.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg
import warnings
warnings.filterwarnings('ignore')

## **3. QR Decomposition**

NumPy provides a convenient function to compute the QR decomposition of a matrix. The function np.linalg.qr returns both the orthogonal matrix Q and the upper-triangular matrix R such that the original matrix can be reconstructed by multiplying them together. Let's examine the structure and properties of these decomposed matrices through a concrete example.

In [2]:
# Create a random matrix
A = np.random.randn(6, 6)

# Compute QR decomposition
Q, R = np.linalg.qr(A)

# Verify that A = QR
reconstructed = Q @ R
reconstruction_error = np.linalg.norm(A - reconstructed)

print("Original matrix A:")
print(A)
print(f"\nReconstruction error (should be near zero): {reconstruction_error:.2e}")

# Verify that Q is orthogonal: Q^T @ Q = I
orthogonality_error = np.linalg.norm(Q.T @ Q - np.eye(6))
print(f"Orthogonality error (Q^T Q - I, should be near zero): {orthogonality_error:.2e}")

# Verify that R is upper-triangular by checking lower triangle
lower_triangle = np.tril(R, k=-1)
lower_triangle_norm = np.linalg.norm(lower_triangle)
print(f"Norm of lower triangle of R (should be near zero): {lower_triangle_norm:.2e}")

Original matrix A:
[[ 0.83149359 -0.29259066  1.12273818  2.11930227 -0.65893636  0.80501821]
 [ 1.26871855  0.89211362 -0.02009713  0.15387189  0.71413814 -0.14420223]
 [ 0.1675868  -2.30468379 -0.43194623  0.08676962  1.10090167 -0.77076172]
 [-0.31558017  2.31410718  0.91851077  2.12483936 -0.20218099 -0.83459445]
 [ 1.30478282  1.10680482  0.04103439 -2.08861041  0.52054284 -2.30043381]
 [ 0.14560543  1.52165981 -0.5389391  -0.75398599 -1.05146138 -0.79203237]]

Reconstruction error (should be near zero): 1.46e-15
Orthogonality error (Q^T Q - I, should be near zero): 7.04e-16
Norm of lower triangle of R (should be near zero): 0.00e+00


### **3.1 Sizes of Q and R Matrices**

The dimensions of the Q and R matrices from QR decomposition depend on the shape of the input matrix A and whether we compute the economy (reduced) or full (complete) QR decomposition. For a tall matrix A with dimensions M × N where M > N, the economy mode QR decomposition produces a Q matrix of size M × N and an R matrix of size N × N. This is the most common and efficient form since we only compute the necessary orthogonal vectors that span the column space of A. In contrast, the full QR decomposition produces a Q matrix of size M × M, which means it includes additional orthogonal vectors that span the null space orthogonal to the column space of A. The full decomposition is sometimes needed for theoretical purposes but is computationally more expensive. For square matrices (M = N), economy and full modes produce the same result. Understanding these size variations is important because it affects how the matrices are used in subsequent computations, especially when solving least squares problems where the choice between economy and full mode matters.

In [3]:
# Tall rectangular matrix
A_tall = np.random.randn(10, 4)

# Economy mode QR decomposition (default)
Q_economy, R_economy = np.linalg.qr(A_tall)

print("Tall matrix A shape:", A_tall.shape)
print("\nEconomy mode QR decomposition:")
print(f"  Q shape: {Q_economy.shape}")
print(f"  R shape: {R_economy.shape}")

# Full mode QR decomposition
Q_full, R_full = np.linalg.qr(A_tall, mode='complete')

print("\nFull mode QR decomposition:")
print(f"  Q shape: {Q_full.shape}")
print(f"  R shape: {R_full.shape}")

# Verify reconstruction still works with economy mode
reconstructed_economy = Q_economy @ R_economy
error_economy = np.linalg.norm(A_tall - reconstructed_economy)
print(f"\nReconstruction error (economy mode): {error_economy:.2e}")

Tall matrix A shape: (10, 4)

Economy mode QR decomposition:
  Q shape: (10, 4)
  R shape: (4, 4)

Full mode QR decomposition:
  Q shape: (10, 10)
  R shape: (10, 4)

Reconstruction error (economy mode): 1.72e-15


### **3.2 Why R is Upper-Triangular**

The upper-triangular structure of the R matrix is a fundamental property of QR decomposition and deserves careful explanation. The key to understanding this lies in the formula R = Q^T A and the way the Gram-Schmidt procedure orthogonalizes vectors. When we compute Q^T A, we are computing dot products between the rows of Q^T (which are the columns of Q) and the columns of A. The critical observation is that the columns of Q are created in order from left to right through the Gram-Schmidt procedure. Each column q_k is orthogonalized against all previous columns q_1 through q_(k-1), but it is not orthogonalized against subsequent columns. This means that when we compute the dot product of q_k with column a_j where j < k (a column that appears earlier in A), we get a nonzero result because a_j has not been orthogonalized against q_k. Conversely, when we compute the dot product of q_k with column a_j where j > k (a column that appears later in A), we might get zero or nonzero depending on whether a_j happens to be orthogonal to q_k by chance. This leads to the pattern where the R matrix has nonzero elements in the upper triangle and mostly zeros in the lower triangle. Furthermore, if the original matrix A already consisted of orthogonal columns, then R would be a diagonal matrix with the norms of each column on the diagonal, since orthogonal columns would produce zero dot products everywhere except on the diagonal where we get the column norms.

In [4]:
# Create a simple example matrix
A_example = np.array([[1, -1], [2, 3], [-2, 2]], dtype=float)

# Compute QR decomposition
Q, R = np.linalg.qr(A_example)

print("Original matrix A:")
print(A_example)

print("\nOrthogonal matrix Q:")
print(Q)

print("\nUpper-triangular matrix R:")
print(R)

# Show that lower triangle of R is zero
print("\nLower triangle of R (should be all zeros or near-zeros):")
print(np.tril(R, k=-1))

# Verify that R = Q^T @ A
R_computed = Q.T @ A_example
print("\nR computed from Q^T @ A:")
print(R_computed)
print(f"\nDifference between computed R and returned R: {np.linalg.norm(R - R_computed):.2e}")

Original matrix A:
[[ 1. -1.]
 [ 2.  3.]
 [-2.  2.]]

Orthogonal matrix Q:
[[-0.33333333  0.2981424 ]
 [-0.66666667 -0.74535599]
 [ 0.66666667 -0.59628479]]

Upper-triangular matrix R:
[[-3.         -0.33333333]
 [ 0.         -3.72677996]]

Lower triangle of R (should be all zeros or near-zeros):
[[0. 0.]
 [0. 0.]]

R computed from Q^T @ A:
[[-3.         -0.33333333]
 [ 0.         -3.72677996]]

Difference between computed R and returned R: 8.95e-16


### **3.3 QR Decomposition and Matrix Inversion**

QR decomposition provides a numerically more stable method for computing the matrix inverse compared to traditional methods. The key insight is that from the decomposition A = QR, we can derive the inverse by inverting both sides of the equation and applying the LIVE EVIL rule (a mnemonic for (AB)^(-1) = B^(-1) A^(-1)). This gives us A^(-1) = (QR)^(-1) = R^(-1) Q^(-1). Since Q is orthogonal, we know that Q^(-1) = Q^T, which means A^(-1) = R^(-1) Q^T. This is advantageous because computing the inverse of an upper-triangular matrix R is numerically much more stable than computing the inverse of a general dense matrix. The process of inverting a triangular matrix can be done efficiently through back-substitution, which is a highly stable numerical procedure. Furthermore, since Q is computed using stable algorithms like Householder reflections, the overall process of inverting A through QR decomposition tends to be more stable than traditional matrix inversion methods. However, it's important to remember that while QR decomposition improves numerical stability, it cannot salvage the inversion of matrices that are nearly singular or have high condition numbers; a poorly conditioned matrix is difficult to invert accurately regardless of the algorithm used.

In [5]:
# Create a random square matrix
A = np.random.randn(5, 5)

# Compute QR decomposition
Q, R = np.linalg.qr(A)

# Compute inverse using QR decomposition
# A^(-1) = R^(-1) @ Q^T
A_inv_qr = np.linalg.inv(R) @ Q.T

# Compute inverse using direct method
A_inv_direct = np.linalg.inv(A)

# Verify both methods give same result
print("Difference between QR-based inverse and direct inverse:")
print(f"{np.linalg.norm(A_inv_qr - A_inv_direct):.2e}")

# Verify that A @ A^(-1) = I
product_qr = A @ A_inv_qr
product_direct = A @ A_inv_direct

print(f"\nError in A @ A^(-1) = I (QR method): {np.linalg.norm(product_qr - np.eye(5)):.2e}")
print(f"Error in A @ A^(-1) = I (direct method): {np.linalg.norm(product_direct - np.eye(5)):.2e}")

Difference between QR-based inverse and direct inverse:
1.46e-15

Error in A @ A^(-1) = I (QR method): 7.92e-16
Error in A @ A^(-1) = I (direct method): 6.29e-16


## **4. Summary**

Orthogonal matrices and QR decomposition represent some of the most important tools in numerical linear algebra and matrix computations. An orthogonal matrix Q possesses the elegant property that Q^T Q = I, which means its transpose is also its inverse. This property makes orthogonal matrices numerically stable and efficient to work with computationally. QR decomposition decomposes any matrix A into the product A = QR, where Q is orthogonal and R is upper-triangular. This decomposition is achieved through the Gram-Schmidt orthogonalization procedure, although modern implementations use more stable algorithms like Householder reflections for actual numerical computations. The upper-triangular structure of R arises naturally from the column-by-column orthogonalization process inherent in Gram-Schmidt. QR decomposition has numerous applications including solving least squares problems, computing matrix inverses more stably than direct inversion, and finding eigenvalues. The key advantages of QR decomposition are its numerical stability and its applicability to any matrix, not just square ones. Understanding orthogonal matrices and QR decomposition is fundamental to advanced linear algebra and numerical computing, making them essential topics for anyone working with data science, machine learning, or scientific computing.